# Upgrade Databricks SDK

This cell upgrades the Databricks SDK to ensure all features (Apps, Lakebase, etc.) are available.

In [0]:
%pip install --upgrade databricks-sdk>=0.61.0
dbutils.library.restartPython()

## Cell 1: Setup and Helper Functions
This notebook Init setup

In [0]:
%python
# Title: Setup and Helper Functions
import time
from datetime import datetime

def run_notebook_with_error_display(notebook_path, timeout_seconds=3600):
    """
    Run a notebook and display detailed error information if it fails.
    Returns: (success: bool, result: dict, error_message: str)
    """
    print(f"\n{'='*80}")
    print(f"🚀 Starting: {notebook_path}")
    print(f"{'='*80}")
    
    start_time = time.time()
    
    try:
        result = dbutils.notebook.run(
            notebook_path, 
            timeout_seconds=timeout_seconds
        )
        
        elapsed = time.time() - start_time
        print(f"✅ SUCCESS: {notebook_path}")
        print(f"⏱️  Completed in {elapsed:.2f} seconds")
        print(f"📊 Result: {result}")
        
        return True, result, None
        
    except Exception as e:
        elapsed = time.time() - start_time
        error_msg = str(e)
        
        print(f"\n{'='*80}")
        print(f"❌ FAILED: {notebook_path}")
        print(f"{'='*80}")
        print(f"⏱️  Failed after {elapsed:.2f} seconds")
        print(f"\n📋 Error Details:")
        print(f"{'-'*80}")
        print(error_msg)
        print(f"{'-'*80}")
        
        # Try to extract more detailed error information
        if "ExecutionError" in error_msg:
            print("\n🔍 This appears to be a notebook execution error.")
            print("   Check the target notebook for detailed error messages.")
        
        return False, None, error_msg

def print_summary(results):
    """Print a summary of all notebook executions"""
    print(f"\n\n{'='*80}")
    print(f"📊 EXECUTION SUMMARY")
    print(f"{'='*80}\n")
    
    total = len(results)
    successful = sum(1 for r in results if r['success'])
    failed = total - successful
    
    for result in results:
        status = "✅ SUCCESS" if result['success'] else "❌ FAILED"
        print(f"{status}: {result['notebook']}")
        if not result['success']:
            print(f"   Error: {result['error'][:100]}...")
    
    print(f"\n{'='*80}")
    print(f"Total: {total} | Success: {successful} | Failed: {failed}")
    print(f"{'='*80}\n")
    
    return successful == total

## Cell 2: Start - Run All Setup Notebooks
Running setup notebooks

In [0]:
# Title: Start - Run All Setup Notebooks
from datetime import datetime
import time

print(f"\n🎬 STARTING DIGITAL TWIN APPLICATION SETUP")
print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Define the notebook execution sequence
notebooks = [
    "./0-Parameters",
    "./1-Create-Sensor-Bronze-Table",
    "./3-Setup-Mapping-Pipeline",
    "./4-Sync-To-Lakebase",
    "./5-Create-App"
]

# Track results
execution_results = []
overall_success = True

# Execute each notebook in sequence
for notebook_path in notebooks:
    success, result, error = run_notebook_with_error_display(notebook_path)
    
    execution_results.append({
        'notebook': notebook_path,
        'success': success,
        'result': result,
        'error': error
    })
    
    # Stop execution if a notebook fails
    if not success:
        overall_success = False
        print(f"\n⚠️  STOPPING EXECUTION DUE TO FAILURE IN: {notebook_path}")
        print(f"⚠️  Remaining notebooks will not be executed.")
        break
    
    # Small delay between notebooks
    time.sleep(2)

# Print summary
print_summary(execution_results)

# Final status
if overall_success:
    print("🎉 ALL NOTEBOOKS COMPLETED SUCCESSFULLY!")
    print("✅ Digital Twin Application setup is complete.")
    print("\n📝 Next Steps:")
    print("   1. Manually deploy the app:")
    print("      - Go to: Compute -> Apps")
    print("      - Find: digitaltwinapp-sk")
    print("      - Click 'Deploy' button")
    print("      - Select path: deployment-staging")
    print("   2. Run '0-App Verify' notebook to check all resources")
else:
    print("❌ SETUP FAILED - Please review errors above and fix issues.")
    print("💡 Tip: Run the failed notebook manually to see detailed error messages.")

print(f"\n⏰ Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")